In [ ]:
!pip install -q --no-deps boltz==2.0.3
!pip install -q "torch>=2.2" "numpy>=1.26" hydra-core==1.3.2 pytorch-lightning==2.5.0 "rdkit>=2024.3.2" requests==2.32.3 "pandas>=2.2.2" types-requests einops==0.8.0 einx==0.3.0 fairscale==0.4.13 mashumaro==3.14 modelcif==1.2 wandb==0.18.7 click==8.1.7 pyyaml==6.0.2 biopython==1.84 scipy==1.13.1 numba==0.61.0 gemmi==0.6.5 scikit-learn==1.6.1 chembl_structure_pipeline==1.2.2 "trifast>=0.1.11"
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "cuda:", torch.cuda.is_available())


In [ ]:
TARGET = 'KKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCKLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLRGC'
DESIGNS = {'r03_l07': 'SVTVDTYADPGDAPMVDHMAVTLGDHVEEAEVFTTENADITVRYKDKHGKVHEHTFTAPPEGTEYTVEIP', 'r03_l05': 'SVTVDTAAEPGDTPVVDHATVTLGDHVEEAEVTVAENGTITVRYKDKHGKVHEHTFTAPPEGTAYTVEIP', 'r01_l07': 'FADCESMFDFPANSAHTVEAPHGSKVIDSCLEEGPDSKTLKITFINEHGEVKETTLTVKKDESFCFD', 'r02_l07': 'DLYEQAEALAKEGKVEEALELLRKLIEEHPADLFHAILDLTIEPNGKKILEELAKEHPELKELIDEALLLN'}

import os
os.makedirs("/kaggle/working/yamls", exist_ok=True)
for name, bseq in DESIGNS.items():
    yaml = f"""version: 1
sequences:
  - protein:
      id: A
      sequence: {bseq}
      msa: empty
  - protein:
      id: B
      sequence: {TARGET}
      msa: empty
"""
    p = f"/kaggle/working/yamls/{name}.yaml"
    open(p, "w").write(yaml)
    print("wrote", p, f"binder={len(bseq)} target={len(TARGET)}")


In [ ]:
import os, time, traceback
os.makedirs("/kaggle/working/preds", exist_ok=True)
import boltz.main as bm
from pytorch_lightning import Trainer as PLTrainer
class NoLogTrainer(PLTrainer):
    def __init__(self, *a, **k):
        k["logger"] = False
        super().__init__(*a, **k)
bm.Trainer = NoLogTrainer
print("Trainer patched: logger disabled", flush=True)
from click.testing import CliRunner
runner = CliRunner()
from importlib import import_module
DESIGNS_ = {'r03_l07': 'SVTVDTYADPGDAPMVDHMAVTLGDHVEEAEVFTTENADITVRYKDKHGKVHEHTFTAPPEGTEYTVEIP', 'r03_l05': 'SVTVDTAAEPGDTPVVDHATVTLGDHVEEAEVTVAENGTITVRYKDKHGKVHEHTFTAPPEGTAYTVEIP', 'r01_l07': 'FADCESMFDFPANSAHTVEAPHGSKVIDSCLEEGPDSKTLKITFINEHGEVKETTLTVKKDESFCFD', 'r02_l07': 'DLYEQAEALAKEGKVEEALELLRKLIEEHPADLFHAILDLTIEPNGKKILEELAKEHPELKELIDEALLLN'}
for name in DESIGNS_:
    y = f"/kaggle/working/yamls/{name}.yaml"
    t0 = time.time()
    print(f"=== folding {name} ===", flush=True)
    try:
        res = runner.invoke(bm.cli, ["predict", y, "--out_dir", "/kaggle/working/preds",
                                     "--cache", "/kaggle/working/boltz_cache"])
        print(f"exit_code={res.exit_code} time={time.time()-t0:.0f}s", flush=True)
        if res.output:
            print(res.output[-2000:], flush=True)
        if res.exception:
            print("EXCEPTION:", flush=True)
            traceback.print_exception(res.exception)
    except Exception:
        traceback.print_exc()
print("ALL DONE", flush=True)


In [ ]:
import json, glob
DESIGNS_ = {'r03_l07': 'SVTVDTYADPGDAPMVDHMAVTLGDHVEEAEVFTTENADITVRYKDKHGKVHEHTFTAPPEGTEYTVEIP', 'r03_l05': 'SVTVDTAAEPGDTPVVDHATVTLGDHVEEAEVTVAENGTITVRYKDKHGKVHEHTFTAPPEGTAYTVEIP', 'r01_l07': 'FADCESMFDFPANSAHTVEAPHGSKVIDSCLEEGPDSKTLKITFINEHGEVKETTLTVKKDESFCFD', 'r02_l07': 'DLYEQAEALAKEGKVEEALELLRKLIEEHPADLFHAILDLTIEPNGKKILEELAKEHPELKELIDEALLLN'}
for name in DESIGNS_:
    pat = f"/kaggle/working/preds/boltz_results_{name}/predictions_{name}/confidence_*_model_0.json"
    files = glob.glob(pat)
    print(name, "->", files)
    for f in files:
        c = json.load(open(f))
        keys = {k: (round(v,4) if isinstance(v,float) else v) for k,v in c.items()
                if k in ("iptm","ptm","ligand_iptm","protein_iptm","complex_plddt","complex_piplddt")}
        print("  ", keys)


In [ ]:
import shutil
shutil.make_archive("/kaggle/working/boltz2_pathB_folds", "zip", "/kaggle/working/preds")
print("zipped -> /kaggle/working/boltz2_pathB_folds.zip")
